# 06 · CASCADE with that same policy
**20 minutes. Objective:** implement a sensor-only student adapter, then (only when admitted and instructor-approved) run one actual local Qwen/CASCADE attempt against the exact Notebook04 actor in a fresh Isaac Sim episode.

CASCADE chooses tools; the instructor-owned ROS/native domains enforce authority; the trained actor actuates joints; the independent evaluator judges motion/rest. A model's success claim is not physical evidence.

The **0.5-second capture and independent 0.5-second lease**, fixed-step/rest limits, BAM and no-motion-retry rules are unchanged. STOP latches; it is neither proof of rest nor torque-off. No world pose, raw health, lease secret, overview image or arbitrary numeric motion tool becomes model input.

In [ ]:
from pathlib import Path
import os, sys, json
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select the host Python 3.12 kernel created by scripts/notebook_setup.sh")
# This kernel is deliberately not a ROS / Kit interpreter.
start = Path(os.environ.get("WORKSHOP_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "notebooks/workshop.py").is_file()
             and (p / "src/microduck_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository, or set WORKSHOP_ROOT")
sys.path.insert(0, str(ROOT / "notebooks"))
from policy_workshop import PolicySession, training_bundle, admitted_bundle, read_json, sha256
session = PolicySession(ROOT, work=os.environ.get("WORKSHOP_NOTEBOOK_WORK"))
VALIDATE = os.environ.get("WORKSHOP_NOTEBOOK_MODE") == "validate"
print("Host Python:", sys.version.split()[0])
print("Mode:", "safe validation (hardware SKIPPED)" if VALIDATE else "interactive")

## Predict · 2 minutes
After a slow model response, should an earlier GO observation still authorize movement? Where must freshness be rechecked? What should happen when both GO and STOP tags are present?

Find the two TODOs in the actual exercise below. Open `exercises/capstone/tools.py` in Jupyter's file browser and edit that file—not the instructor domain or evaluator. The notebook deliberately does not fill in the answer for you.

In [ ]:
exercise = ROOT / "exercises/capstone/tools.py"
print(exercise.read_text())
RUN_ADAPTER_TESTS = False
USE_REFERENCE = False  # explicit opt-in after the coding timebox, never a hidden fallback
AGENT_IMAGE = os.environ.get("WORKSHOP_AGENT_IMAGE", "roscon/microduck-agent:local")

def selected_tools():
    relative = Path("solutions/capstone/tools.py" if USE_REFERENCE else "exercises/capstone/tools.py")
    print("Selected adapter:", "EXPLICIT INSTRUCTOR REFERENCE" if USE_REFERENCE else "STUDENT IMPLEMENTATION", relative)
    return relative

## Task · 7 minutes: write the adapter
1. `translate_observation`: build a **new allowlisted** dict from valid episode/frame/capture/tag evidence. Effective age includes the full round trip; reject nonfinite/negative ages and malformed counters/IDs. STOP tag1 overrides GO tag0; missing/stale/foreign evidence is not GO. Never copy raw reason/metadata/world pose or secrets.
2. `step_forward`: accept only `{}` or `{"action":"step_forward"}`. Reject all extra/numeric motion arguments rather than clipping them. Delegate once to `domain.step_forward(args)` and preserve its actual receipt. Do not call transport/gateway internals, retry, renew a lease or reset the robot.

Run the actual CPU contract tests. An unfinished student file **must fail**. Read the failing assertion, edit the exercise and rerun CPU tests. These retries are code tests, not motion retries. If time ends, explicitly select `USE_REFERENCE=True` and announce it; your result then uses the reference, not a completed student solution.

In [ ]:
def test_adapter():
    relative = selected_tools()
    session.job("adapter-tests", AGENT_IMAGE,
        ["python", "/course/scripts/agent_check_exercise.py", "--tools-file", "/course/" + relative.as_posix()],
        timeout=90, options=session.course_options())
    state = session.state()
    state["adapter"] = {"path": relative.as_posix(), "sha256": sha256(ROOT / relative),
                        "kind": "EXPLICIT_REFERENCE" if USE_REFERENCE else "STUDENT_IMPLEMENTATION"}
    session.save(state)
    print("CPU contract checks completed; not model/ROS/physics acceptance")
session.step("real student/reference adapter CPU tests", RUN_ADAPTER_TESTS, test_adapter)

## Model readiness · 1 minute (prepared before class)
Keep Notebook05's Sim shut down while staff starts the prepared local Qwen model, especially on L40S. This notebook does not provision a model server or use hosted credentials.

The next probe calls the **real local model** with a clearly labeled read-only synthetic diagnostic. That checks model/tool readiness, not sensor or motion success. It must finish **before** creating the ROS gateway and its one-shot lease. Leave the controls off if the model is unavailable.

In [ ]:
CHECK_MODEL = False
RUN_AGENT = False
INSTRUCTOR_RELEASE_APPROVED = False
ACCEPT_NVIDIA_EULA = False
GPU = os.environ.get("WORKSHOP_GPU", "0")
SIM_IMAGE = os.environ.get("WORKSHOP_SIM_IMAGE", "roscon/microduck-sim:local")
ROS_IMAGE = os.environ.get("WORKSHOP_ROS_IMAGE", "roscon/microduck-ros5:local")
ASSETS = Path(os.environ.get("WORKSHOP_ASSET_ROOT", ROOT / ".cache/assets")).resolve()
MODEL_URL = os.environ.get("WORKSHOP_MODEL_URL", "http://127.0.0.1:8080/v1")
MODEL = "qwen3.8-27b"
SIM_URL = "http://127.0.0.1:8766"

def model_readiness():
    if set(session.state().get("containers", {})) & {"sim", "ros", "agent", "train", "admission"}:
        raise RuntimeError("Explicitly clean up your prior runtime first; model-ready precedes gateway")
    session.capabilities(AGENT_IMAGE, agent=True)
    models = session.get_json(MODEL_URL + "/models")  # loopback-only, no credentials/redirects
    if MODEL not in [row.get("id") for row in models.get("data", [])]:
        raise RuntimeError("Selected local model is not advertised; do not substitute a different model")
    output = session.new_output("model-diagnostic")
    session.job("model-probe", AGENT_IMAGE,
        ["python", "/course/scripts/agent_model_probe.py", "--base-url", MODEL_URL,
         "--model", MODEL, "--out", "/outputs/probe"],
        timeout=210, network="host", options=[*session.course_options(), "-v", f"{output}:/outputs"])
    state = session.state()
    state["model_ready"] = {"base_url": MODEL_URL, "model": MODEL, "result": str(output / "probe/result.json")}
    session.save(state)
session.step("real local model read-only diagnostic before lease", CHECK_MODEL, model_readiness)

## Execute · 5 minutes: exact policy, new episode, one attempt
With staff's explicit release approval, re-hash the complete04/05 bundle and recheck canonical admission. The new native episode must differ from05. An onboard camera frame is displayed for the human; it is not the cached observation used for motion. The domain re-reads real ROS perception after inference.

The gateway starts only in the single-attempt cell, immediately before CASCADE. Stop if any check fails. An expired lease, uncertain delivery or STOP is a refusal—not permission to rerun the motion cell. Re-entering a stopped episode never rearms it.

In [ ]:
def prepare_agent_episode():
    if INSTRUCTOR_RELEASE_APPROVED is not True:
        raise RuntimeError("Staff has not approved the integrated live route")
    bundle = session.selection("05")
    trained = session.selection("04")
    if any(bundle["identity"][key] != value for key, value in trained["identity"].items()):
        raise RuntimeError("Notebook04/05 artifact identity changed")
    adapter = session.state().get("adapter", {})
    relative = selected_tools()
    if adapter.get("path") != relative.as_posix() or adapter.get("sha256") != sha256(ROOT / relative):
        raise RuntimeError("Run CPU tests on this exact selected adapter first")
    ready = session.state().get("model_ready", {})
    if (ready.get("base_url"), ready.get("model")) != (MODEL_URL, MODEL) or not Path(ready.get("result", "")).is_file():
        raise RuntimeError("Complete the actual read-only model diagnostic first")
    session.get_json(MODEL_URL + "/models")
    session.gpu_preflight(GPU, accept_eula=ACCEPT_NVIDIA_EULA)
    session.capabilities(SIM_IMAGE)
    entry = session.start_policy(SIM_IMAGE, bundle, gpu=GPU, assets=ASSETS, url=SIM_URL,
                                 previous_episode=session.require("05")["episode_id"])
    from IPython.display import Image, display
    path = session.camera_snapshot(SIM_URL, expected_episode=entry["episode_id"], label="06-head")
    display(Image(filename=str(path)))
    info = session.get_json(SIM_URL + "/v1/camera_info")
    entry["frame_id"] = info["frame_id"]
    entry["adapter"] = adapter
    state = session.state()
    state["agent_episode"] = entry
    session.save(state)
    print("Supervisor-only exact-policy identity:", session.live_identity(SIM_URL, bundle, entry["episode_id"]))
session.step("fresh admitted student-policy episode after model readiness", RUN_AGENT, prepare_agent_episode)

In [ ]:
def run_one_agent_attempt():
    if INSTRUCTOR_RELEASE_APPROVED is not True:
        raise RuntimeError("Instructor approval required")
    entry = session.state()["agent_episode"]
    bundle = session.selection("05")
    if bundle["identity"] != entry["identity"]:
        raise RuntimeError("Policy identity changed before gateway")
    relative = selected_tools()
    if entry["adapter"]["path"] != relative.as_posix() or entry["adapter"]["sha256"] != sha256(ROOT / relative):
        raise RuntimeError("Adapter changed; no motion, repeat CPU validation first")
    session.get_json(MODEL_URL + "/models")  # model ready before acquiring any gateway lease
    session.live_identity(SIM_URL, bundle, entry["episode_id"])
    session.wait_sensors(SIM_URL, expected_episode=entry["episode_id"])  # read-only; no gateway lease yet
    session.claim_attempt(entry["episode_id"])  # persisted BEFORE dispatch, including uncertain launch failures
    output = session.new_output("agent")
    state = session.state()
    state["agent_attempt"] = {"output": str(output / "attempt"), "episode_id": entry["episode_id"],
                              "identity": bundle["identity"], "adapter": entry["adapter"]}
    session.save(state)
    try:
        session.start("ros", ROS_IMAGE,
            ["ros2", "launch", "microduck_workshop_ros", "workshop.launch.py",
             "sim_url:=" + SIM_URL, "tag_size:=0.20"], network="host", memory="8g", cpus=4,
            options=[*session.gpu_options(GPU), "--ipc", "shareable", "--shm-size", "1g",
                     "-e", "RMW_IMPLEMENTATION=rmw_fastrtps_cpp"])
        session.wait_ros_node("ros", "/workshop/microduck/gateway", timeout=60)
        code, _ = session.job("agent", AGENT_IMAGE,
            ["python", "-m", "microduck_workshop.agent.run", "--base-url", MODEL_URL, "--model", MODEL,
             "--episode-id", entry["episode_id"], "--frame-id", entry["frame_id"],
             "--run-dir", "/outputs/attempt", "--max-steps", "6",
             "--tools-file", "/course/" + relative.as_posix(),
             "--policy-manifest", "/policy-run/policy_manifest.json",
             "--policy-admission", "/policy-admission/admission.json",
             "--native-health-url", SIM_URL + "/v1/health"],
            timeout=420, allowed=(0, 2), network="host",
            options=[*session.course_options(), *session.bundle_options(bundle), "-v", f"{output}:/outputs",
                     "--ipc", "container:" + session.name("ros"), "-e", "RMW_IMPLEMENTATION=rmw_fastrtps_cpp"])
        print("Actual agent exit:", code, "(0 is an agent report, 2 is refusal/failure; neither replaces physical audit)")
    finally:
        session.stop("ros")  # no surviving lease client after an exception or timeout
session.step("single real CASCADE/ROS attempt, no retry", RUN_AGENT, run_one_agent_attempt)

## Observe · 3 minutes
Read the actual provider transcript, trace, report and independent native verdicts. Distinguish **accepted**, **completed**, **verification_available** and **verified**. `accepted:true` is not a completed or physically verified maneuver. A missing report after an uncertain send remains missing; do not fabricate it or retry.

The runner checks live identity before client creation and after the model run. The notebook independently compares it again and retains the04 checkpoint hash. Neither raw health nor evaluator truth is sent to the model. Native verdict reading below is **offline human audit only**, never a control input.

In [ ]:
def inspect_actual_attempt():
    entry = session.state()["agent_episode"]
    attempt = session.state()["agent_attempt"]
    bundle = session.selection("05")
    trained = session.selection("04")
    if any(bundle["identity"][key] != value for key, value in trained["identity"].items()):
        raise RuntimeError("Trained artifact changed after episode")
    session.live_identity(SIM_URL, bundle, entry["episode_id"], require_unstopped=False)
    directory = Path(attempt["output"])
    report = read_json(directory / "report.json")
    identity = read_json(directory / "policy_identity.json")
    expected = {key: bundle["identity"][key] for key in ("policy_sha256", "manifest_sha256", "admission_sha256")}
    if any(identity.get(key) != value for key, value in expected.items()) or identity.get("episode_id") != entry["episode_id"]:
        raise RuntimeError("Agent receipt policy/episode mismatch")
    if report.get("selected_policy_identity") != identity:
        raise RuntimeError("Agent final report does not preserve selected identity")
    print("Adapter provenance:", attempt["adapter"])
    print("Exact checkpoint through CASCADE:", trained["identity"]["checkpoint_sha256"])
    print("Actual agent report (not independent physical acceptance):", json.dumps(report, indent=2))
    verdict_file = Path(entry["output"]) / "episode/evaluator_verdicts.jsonl"
    verdicts = [json.loads(line) for line in verdict_file.read_text().splitlines()] if verdict_file.is_file() else []
    print("Actual independent native verdicts (empty means none observed):", json.dumps(verdicts, indent=2))
    files = [*bundle["files"], directory / "report.json", directory / "provider.jsonl", directory / "policy_identity.json"]
    # Keep the mutable live native log separate; cleanup below flushes the simulator.
    session.complete("06", {**attempt, "kind": "ACTUAL_AGENT_ATTEMPT_REQUIRES_INDEPENDENT_AUDIT"}, artifacts=files)
session.step("audit actual report and same-policy identity", RUN_AGENT, inspect_actual_attempt)

## Debug and Cleanup · 2 minutes
- Unfinished adapter: expected CPU test failure. Implement it or openly select the reference; no hidden substitution.
- Failed admission/model readiness: stay with CPU/no-motion analysis. Do not remove the policy flags or fall back to the public actor.
- Lease/capture expiry or gateway stall: keep timings/logs, refuse motion and ask staff. Do not widen either0.5-second bound, auto-renew, or automatically rearm.
- Agent timeout/uncertain send: the attempt is consumed. No motion retry. Cleanup is safe to repeat; motion is not.
- STOP/restarted episode: preserve evidence and shut down. A different episode requires deliberate new operator authorization; no warmup STOP followed by automatic reset.

Explain which link your actual evidence proves, and which remains unverified. The final five-minute workshop shutdown also stops your own Jupyter/model supervisor and verifies cloud billing in the authorized console. These cells never stop another team's process or delete paid infrastructure.

In [ ]:
def cleanup_capstone():
    for role in ("agent", "ros", "sim", "model-probe", "adapter-tests"):
        session.stop(role)
session.step("owned capstone cleanup, preserve all evidence", RUN_AGENT or CHECK_MODEL or RUN_ADAPTER_TESTS, cleanup_capstone)